# 03 · Creación del pipeline de ingeniería de características

**Proyecto #2 – Pipeline de Machine Learning · Predict Future Sales (Kaggle)**

Con las decisiones del notebook `02_feature_exploration.ipynb`, aquí se construye **un único
`Pipeline` de scikit-learn** con todos los pasos de ingeniería de características, se ajusta con los
meses de entrenamiento (0-26), se valida y se guarda en **`models/preprocessing_pipeline.pkl`**.

| Punto del enunciado | Sección |
|---|---|
| 2.a Proceso de elaboración del pipeline con la información del paso anterior | 1, 2 |
| 2.b Pipeline con sklearn (usando el operador `Mapper` de clase) | 2 |
| 2.c Archivo `.pkl` con todos los pasos | 4 |

**Diseño clave: el pipeline recibe datos crudos.** La entrada son solo las columnas
`date_block_num, shop_id, item_id` (lo que trae `test.csv` más el mes a predecir). El primer paso,
`SalesHistoryFeatures`, lleva guardada la historia mensual de ventas y genera los lags consultando
solo meses anteriores a cada fila. Por eso el mismo `.pkl` sirve para entrenar, validar y predecir
noviembre 2015 sin ninguna preparación manual previa.

Los transformadores propios (`SalesHistoryFeatures`, `Mapper`, `DtypeDowncaster`) están definidos en
`predict_future_sales/features.py` y no en el notebook: `pickle` guarda la *ruta* de la clase, así que
si estuvieran aquí el `.pkl` no se podría cargar desde otro notebook o script.

## 0. Configuración

In [1]:
import os

os.environ["LOGURU_LEVEL"] = "WARNING"

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

import time

from feature_engine.encoding import MeanEncoder, OneHotEncoder
from feature_engine.imputation import AddMissingIndicator, CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.selection import DropFeatures
from feature_engine.transformation import LogCpTransformer
import joblib
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from predict_future_sales.config import MODELS_DIR, PROCESSED_DATA_DIR
from predict_future_sales.dataset import load_raw_data
import predict_future_sales.features as pf

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 50)
set_config(display="text")  # representación de texto del pipeline (legible también en GitHub)
TRAIN_END, VAL_START = 26, 27
PIPELINE_PATH = MODELS_DIR / "preprocessing_pipeline.pkl"

## 1. Datos de entrada

Mismo dataset base del notebook 02: grid mensual tienda × producto con ceros (objetivo recortado a
[0, 20]), construido desde `data/processed/monthly_shop_item.csv` del notebook 01. El pipeline se
ajusta con la partición secuencial del enunciado: **entrenamiento = meses 0-26**.

In [2]:
monthly = pd.read_csv(PROCESSED_DATA_DIR / "monthly_shop_item.csv")
raw = load_raw_data()
maps = pf.build_mappings(raw["items"], raw["categories"], raw["shops"])
grid = pf.build_monthly_grid(monthly)

train = grid[grid.date_block_num <= TRAIN_END]
val = grid[grid.date_block_num >= VAL_START]
X_train, y_train = train[pf.KEYS], train[pf.TARGET]
X_val, y_val = val[pf.KEYS], val[pf.TARGET]
X_test = raw["test"].assign(date_block_num=pf.TEST_BLOCK)[pf.KEYS]  # test.csv crudo + mes 34

pd.DataFrame(
    {"filas": [len(X_train), len(X_val), len(X_test)],
     "meses": ["0-26", "27-33", "34"],
     "columnas de entrada": [list(X_train.columns)] * 3},
    index=["entrenamiento", "validación", "test"],
)

,filas,meses,columnas de entrada
entrenamiento,9299440,0-26,"[date_block_num, shop_id, item_id]"
validación,1614364,27-33,"[date_block_num, shop_id, item_id]"
test,214200,34,"[date_block_num, shop_id, item_id]"


## 2. Construcción del pipeline

Cada paso corresponde a una fila de la tabla 4.2 del notebook 02. Las listas de variables son las
constantes de `predict_future_sales/features.py`, para que notebooks y scripts usen exactamente las
mismas definiciones.

| Paso | Proceso del enunciado | Operador |
|---|---|---|
| `history` | generación de lags y variables exógenas | `SalesHistoryFeatures` (propio) |
| `map_city`, `map_shop_type`, `map_category_group` | mapeo (operador de clase) | `Mapper` (propio) |
| `drop_keys` | selección | `DropFeatures` |
| `missing_indicator`, `num_imputer` | i. imputación numérica | `AddMissingIndicator`, `MeanMedianImputer` |
| `cat_imputer` | ii. imputación categórica | `CategoricalImputer` |
| `onehot`, `mean_encoder` | iii. codificación | `OneHotEncoder`, `MeanEncoder` |
| `outliers` | iv. outliers | `Winsorizer` |
| `log` | v. transformación | `LogCpTransformer` |
| `scaler` | vi. escalado | `StandardScaler` |
| `float32` | memoria | `DtypeDowncaster` (propio) |

In [3]:
preprocessing_pipeline = Pipeline(
    steps=[
        # Lags (par, producto, categoría, tienda, tienda×categoría), precio, antigüedad, banderas y
        # calendario. Solo consulta meses < date_block_num de cada fila: sin fuga de información.
        ("history", pf.SalesHistoryFeatures(
            history=monthly, items=raw["items"],
            pair_lags=pf.PAIR_LAGS, item_lags=pf.ITEM_LAGS,
            cat_lags=pf.GROUP_LAGS, shop_lags=pf.GROUP_LAGS, shop_cat_lags=pf.GROUP_LAGS,
        )),
        # Mapeos con diccionarios (operador Mapper): atributos de tienda y de categoría
        ("map_city", pf.Mapper(variables=["shop_id"], mappings=maps["city"], new_variables=["city"])),
        ("map_shop_type", pf.Mapper(variables=["shop_id"], mappings=maps["shop_type"], new_variables=["shop_type"])),
        ("map_category_group", pf.Mapper(variables=["item_category_id"], mappings=maps["category_group"],
                                         new_variables=["category_group"])),
        # Llaves que no entran al modelo: mes absoluto (no extrapolable) e id crudo del producto
        ("drop_keys", DropFeatures(features_to_drop=pf.DROP_VARS)),
        # i. Imputación numérica: bandera de "sin historia" + mediana de entrenamiento
        ("missing_indicator", AddMissingIndicator(variables=pf.MISSING_INDICATOR_VARS)),
        ("num_imputer", MeanMedianImputer(imputation_method="median", variables=pf.NUM_IMPUTE_VARS)),
        # ii. Imputación categórica: categoría propia para valores sin mapeo
        ("cat_imputer", CategoricalImputer(imputation_method="missing", fill_value="Missing", variables=pf.CAT_VARS)),
        # iii. Codificación: one-hot (baja cardinalidad) y mean encoding suavizado
        ("onehot", OneHotEncoder(variables=pf.ONEHOT_VARS, drop_last=True)),
        ("mean_encoder", MeanEncoder(variables=pf.MEAN_ENCODE_VARS, smoothing="auto", unseen="encode")),
        # iv. Outliers: capping del precio en el percentil 99 de entrenamiento
        ("outliers", Winsorizer(capping_method="quantiles", tail="right", fold=0.01, variables=pf.OUTLIER_VARS)),
        # v. Transformación log(x + 1) de conteos y precio
        ("log", LogCpTransformer(variables=pf.LOG_VARS, C=1)),
        # vi. Escalado: media 0, desviación 1 (necesario para el modelo lineal)
        ("scaler", StandardScaler().set_output(transform="pandas")),
        # float32: la mitad de memoria (~10 millones de filas)
        ("float32", pf.DtypeDowncaster()),
    ]
)


def describe_step(step):
    # Variables sobre las que actúa cada paso (los transformadores de feature-engine usan `variables`)
    if isinstance(step, pf.Mapper):
        return f"{step.variables[0]} → {step.new_variables[0]} ({len(step.mappings)} claves)"
    for attr in ("variables", "features_to_drop"):
        cols = getattr(step, attr, None)
        if cols:
            return ", ".join(cols) if len(cols) <= 4 else f"{len(cols)} variables: {', '.join(cols[:3])}, …"
    if isinstance(step, pf.SalesHistoryFeatures):
        return f"entrada date_block_num, shop_id, item_id · lags par {step.pair_lags}, producto {step.item_lags}"
    return "todas las columnas"


pd.DataFrame(
    [{"paso": name, "operador": type(step).__name__, "actúa sobre": describe_step(step)}
     for name, step in preprocessing_pipeline.steps],
    index=pd.RangeIndex(1, len(preprocessing_pipeline.steps) + 1, name="#"),
)

,paso,operador,actúa sobre
#,,,
1,history,SalesHistoryFeatures,"entrada date_block_num, shop_id, item_id · lag..."
2,map_city,Mapper,shop_id → city (60 claves)
3,map_shop_type,Mapper,shop_id → shop_type (60 claves)
4,map_category_group,Mapper,item_category_id → category_group (84 claves)
5,drop_keys,DropFeatures,"date_block_num, item_id"
6,missing_indicator,AddMissingIndicator,"6 variables: cnt_lag_1, cnt_lag_2, cnt_lag_3, …"
7,num_imputer,MeanMedianImputer,"20 variables: cnt_lag_1, cnt_lag_2, cnt_lag_3, …"
8,cat_imputer,CategoricalImputer,"city, shop_type, category_group"
9,onehot,OneHotEncoder,shop_type


## 3. Ajuste y validación

El pipeline se ajusta **solo con los meses 0-26**: medianas, medias por ciudad / grupo, percentil 99
del precio y parámetros del escalado salen de entrenamiento y se aplican sin cambios a validación y
test. (`MeanEncoder` es el único paso que usa `y`.)

In [4]:
t0 = time.perf_counter()
Xt_train = preprocessing_pipeline.fit_transform(X_train, y_train)
t_fit = time.perf_counter() - t0
t0 = time.perf_counter()
Xt_val = preprocessing_pipeline.transform(X_val)
Xt_test = preprocessing_pipeline.transform(X_test)
t_tr = time.perf_counter() - t0
print(f"fit_transform entrenamiento: {t_fit:.0f} s · transform validación + test: {t_tr:.0f} s")
print(f"Entrenamiento {Xt_train.shape} · validación {Xt_val.shape} · test {Xt_test.shape}")
print(f"Memoria de la matriz de entrenamiento: {Xt_train.memory_usage().sum() / 1e9:.2f} GB (float32)")
Xt_train.head()

fit_transform entrenamiento: 42 s · transform validación + test: 5 s
Entrenamiento (9299440, 40) · validación (1614364, 40) · test (214200, 40)
Memoria de la matriz de entrenamiento: 1.49 GB (float32)


,shop_id,item_category_id,cnt_lag_1,cnt_lag_2,cnt_lag_3,cnt_lag_6,cnt_lag_12,item_cnt_lag_1,item_cnt_lag_2,item_cnt_lag_3,item_cnt_lag_6,item_cnt_lag_12,cat_cnt_lag_1,cat_cnt_lag_2,cat_cnt_lag_12,shop_cnt_lag_1,shop_cnt_lag_2,shop_cnt_lag_12,shop_cat_cnt_lag_1,shop_cat_cnt_lag_2,shop_cat_cnt_lag_12,item_price_last,item_is_new,pair_is_new,item_age,pair_age,month,days_in_month,city,category_group,cnt_lag_1_na,cnt_lag_2_na,cnt_lag_3_na,cnt_lag_6_na,cnt_lag_12_na,item_price_last_na,shop_type_Centro comercial,shop_type_Calle,shop_type_Online,shop_type_Itinerante
0,-1.6948,-0.3240,-0.3531,-0.3422,-0.3302,-0.2920,-0.2142,-0.3336,-0.3209,-0.3204,-0.3121,-0.2959,-0.2129,-0.2041,-0.1633,-0.2490,-0.2345,-0.1099,-0.2977,-0.2893,-0.2092,-0.1493,3.0255,0.8783,-1.2311,-0.7966,-1.4197,0.6587,-1.2601,-0.3967,4.9463,3.3987,2.6986,1.7635,1.0352,3.0241,0.3794,-0.3043,-0.2010,-0.0393
1,-1.6948,-1.7367,-0.3531,-0.3422,-0.3302,-0.2920,-0.2142,-0.3336,-0.3209,-0.3204,-0.3121,-0.2959,-0.2129,-0.2041,-0.1633,-0.2490,-0.2345,-0.1099,-0.2977,-0.2893,-0.2092,-0.1493,3.0255,0.8783,-1.2311,-0.7966,-1.4197,0.6587,-1.2601,1.8083,4.9463,3.3987,2.6986,1.7635,1.0352,3.0241,0.3794,-0.3043,-0.2010,-0.0393
2,-1.6948,-0.9967,-0.3531,-0.3422,-0.3302,-0.2920,-0.2142,-0.3336,-0.3209,-0.3204,-0.3121,-0.2959,-0.2129,-0.2041,-0.1633,-0.2490,-0.2345,-0.1099,-0.2977,-0.2893,-0.2092,-0.1493,3.0255,0.8783,-1.2311,-0.7966,-1.4197,0.6587,-1.2601,1.8142,4.9463,3.3987,2.6986,1.7635,1.0352,3.0241,0.3794,-0.3043,-0.2010,-0.0393
3,-1.6948,-1.4676,-0.3531,-0.3422,-0.3302,-0.2920,-0.2142,-0.3336,-0.3209,-0.3204,-0.3121,-0.2959,-0.2129,-0.2041,-0.1633,-0.2490,-0.2345,-0.1099,-0.2977,-0.2893,-0.2092,-0.1493,3.0255,0.8783,-1.2311,-0.7966,-1.4197,0.6587,-1.2601,1.8083,4.9463,3.3987,2.6986,1.7635,1.0352,3.0241,0.3794,-0.3043,-0.2010,-0.0393
4,-1.6948,-0.3240,-0.3531,-0.3422,-0.3302,-0.2920,-0.2142,-0.3336,-0.3209,-0.3204,-0.3121,-0.2959,-0.2129,-0.2041,-0.1633,-0.2490,-0.2345,-0.1099,-0.2977,-0.2893,-0.2092,-0.1493,3.0255,0.8783,-1.2311,-0.7966,-1.4197,0.6587,-1.2601,-0.3967,4.9463,3.3987,2.6986,1.7635,1.0352,3.0241,0.3794,-0.3043,-0.2010,-0.0393


In [5]:
# Validaciones del resultado
checks = {
    "sin NaN en entrenamiento": int(Xt_train.isna().sum().sum()) == 0,
    "sin NaN en validación": int(Xt_val.isna().sum().sum()) == 0,
    "sin NaN en test (mes 34)": int(Xt_test.isna().sum().sum()) == 0,
    "mismas columnas en los 3 conjuntos": list(Xt_train.columns) == list(Xt_val.columns) == list(Xt_test.columns),
    "test conserva las 214,200 filas y su orden": len(Xt_test) == len(raw["test"]) and (Xt_test.index == raw["test"].index).all(),
    "todas las columnas numéricas float32": (Xt_train.dtypes == np.float32).all(),
    "entrenamiento escalado (media ≈ 0, desv. ≈ 1)": bool(np.allclose(Xt_train.mean(), 0, atol=1e-2) and np.allclose(Xt_train.std(), 1, atol=1e-2)),
}
assert all(checks.values()), checks
pd.Series(checks, name="OK").to_frame()

,OK
sin NaN en entrenamiento,True
sin NaN en validación,True
sin NaN en test (mes 34),True
mismas columnas en los 3 conjuntos,True
"test conserva las 214,200 filas y su orden",True
todas las columnas numéricas float32,True
"entrenamiento escalado (media ≈ 0, desv. ≈ 1)",True


In [6]:
# Lista de columnas que recibirá el modelo, agrupadas por origen
groups = {
    "lags del par": [c for c in Xt_train.columns if c.startswith("cnt_lag") and not c.endswith("_na")],
    "lags del producto": [c for c in Xt_train.columns if c.startswith("item_cnt_lag")],
    "lags de categoría / tienda / tienda×categoría": [c for c in Xt_train.columns if c.startswith(("cat_cnt", "shop_cnt", "shop_cat_cnt"))],
    "indicadores de faltante": [c for c in Xt_train.columns if c.endswith("_na")],
    "tienda (exógenas)": ["shop_id", "city"] + [c for c in Xt_train.columns if c.startswith("shop_type_")],
    "producto (exógenas)": ["item_category_id", "category_group", "item_price_last", "item_is_new", "pair_is_new", "item_age", "pair_age"],
    "calendario": ["month", "days_in_month"],
}
assert sorted(sum(groups.values(), [])) == sorted(Xt_train.columns)
pd.DataFrame({"n": {k: len(v) for k, v in groups.items()}, "columnas": {k: ", ".join(v) for k, v in groups.items()}})

,n,columnas
lags del par,5,"cnt_lag_1, cnt_lag_2, cnt_lag_3, cnt_lag_6, cn..."
lags del producto,5,"item_cnt_lag_1, item_cnt_lag_2, item_cnt_lag_3..."
lags de categoría / tienda / tienda×categoría,9,"cat_cnt_lag_1, cat_cnt_lag_2, cat_cnt_lag_12, ..."
indicadores de faltante,6,"cnt_lag_1_na, cnt_lag_2_na, cnt_lag_3_na, cnt_..."
tienda (exógenas),6,"shop_id, city, shop_type_Centro comercial, sho..."
producto (exógenas),7,"item_category_id, category_group, item_price_l..."
calendario,2,"month, days_in_month"


### 3.1 Prueba de no-fuga de información con el pipeline completo

Si el pipeline no usa información futura, las variables de un mes `m` deben ser **idénticas** cuando
la historia que lleva guardada se corta en `m − 1`. Se prueba con el mes 30 (dentro de validación):
se arma una copia del pipeline cuya historia solo llega al mes 29, se le copian los parámetros ya
aprendidos y se comparan las salidas.

In [7]:
from sklearn.base import clone

rows30 = X_val[X_val.date_block_num == 30]
truncated = clone(preprocessing_pipeline)
truncated.steps[0] = ("history", pf.SalesHistoryFeatures(
    history=monthly[monthly.date_block_num < 30], items=raw["items"],
    pair_lags=pf.PAIR_LAGS, item_lags=pf.ITEM_LAGS,
    cat_lags=pf.GROUP_LAGS, shop_lags=pf.GROUP_LAGS, shop_cat_lags=pf.GROUP_LAGS,
).fit(rows30))
for name, step in preprocessing_pipeline.steps[1:]:  # mismos pasos ya ajustados (medianas, encoders, escalado)
    truncated.set_params(**{name: step})
pd.testing.assert_frame_equal(truncated.transform(rows30), preprocessing_pipeline.transform(rows30))
print(f"OK: {len(rows30):,} filas del mes 30 idénticas con historia completa o cortada en el mes 29")

OK: 228,889 filas del mes 30 idénticas con historia completa o cortada en el mes 29


In [8]:
# Parámetros aprendidos en entrenamiento por algunos pasos
steps = preprocessing_pipeline.named_steps
learned = {
    "num_imputer: mediana de cnt_lag_1": steps["num_imputer"].imputer_dict_["cnt_lag_1"],
    "num_imputer: mediana de item_price_last": steps["num_imputer"].imputer_dict_["item_price_last"],
    "outliers: límite de item_price_last (p99)": steps["outliers"].right_tail_caps_["item_price_last"],
    "mean_encoder: Moscú": steps["mean_encoder"].encoder_dict_["city"].get("Moscú"),
    "mean_encoder: grupo Juegos": steps["mean_encoder"].encoder_dict_["category_group"].get("Juegos"),
    "scaler: media de cnt_lag_1 (log)": steps["scaler"].mean_[list(steps["scaler"].feature_names_in_).index("cnt_lag_1")],
}
pd.Series(learned, name="valor aprendido").to_frame()

,valor aprendido
num_imputer: mediana de cnt_lag_1,0.0000
num_imputer: mediana de item_price_last,299.0000
outliers: límite de item_price_last (p99),"4,849.4165"
mean_encoder: Moscú,0.4719
mean_encoder: grupo Juegos,0.6957
scaler: media de cnt_lag_1 (log),0.1331


**Interpretación.** El pipeline se ajusta en unos 40 segundos con 9.3 millones de filas y transforma validación y test en 5 segundos. Produce **40 columnas numéricas `float32`** (1.49 GB en entrenamiento, la mitad de lo que ocuparía en `float64`), sin ningún faltante en los tres conjuntos, con las mismas columnas en el mismo orden y conservando las 214,200 filas de `test.csv` en su orden original (requisito para armar el archivo de envío). La prueba de no-fuga con el pipeline completo pasó: las 228,889 filas del mes 30 son idénticas tanto si el pipeline conoce la historia completa como si solo llega hasta el mes 29. Los parámetros aprendidos son los mismos que se analizaron en el notebook 02: mediana 0 para el lag 1, precio mediano 299 rublos, límite de *capping* en 4,849 rublos (percentil 99) y medias codificadas como Moscú 0.47 o Juegos 0.70 unidades por par y mes.

## 4. Guardar el pipeline (`.pkl`)

Se guarda el pipeline **ajustado** con `joblib` (compresión nivel 3). Incluye la historia mensual de
ventas que necesita `SalesHistoryFeatures`, por lo que es autocontenido. Después se carga de nuevo
desde disco y se comprueba que produce exactamente la misma salida.

In [9]:
PIPELINE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessing_pipeline, PIPELINE_PATH, compress=3)
print(f"Guardado: {PIPELINE_PATH.relative_to(PIPELINE_PATH.parents[1])} ({PIPELINE_PATH.stat().st_size / 1e6:.1f} MB)")

reloaded = joblib.load(PIPELINE_PATH)
pd.testing.assert_frame_equal(reloaded.transform(X_test), Xt_test)
print("OK: el pipeline cargado desde disco reproduce exactamente la salida para test.csv")
print("Pasos:", [name for name, _ in reloaded.steps])

Guardado: models\preprocessing_pipeline.pkl (18.2 MB)


OK: el pipeline cargado desde disco reproduce exactamente la salida para test.csv
Pasos: ['history', 'map_city', 'map_shop_type', 'map_category_group', 'drop_keys', 'missing_indicator', 'num_imputer', 'cat_imputer', 'onehot', 'mean_encoder', 'outliers', 'log', 'scaler', 'float32']


## 5. Conclusiones

1. Se construyó un único `Pipeline` de scikit-learn con **14 pasos**: generación de historia (lags y variables exógenas), 3 `Mapper`, eliminación de llaves y los 6 procesos de ingeniería de características (imputación numérica con indicador, imputación categórica, one-hot y mean encoding, outliers, `log(x + 1)` y escalado), más la conversión a `float32`.
2. El pipeline **acepta datos crudos** (`date_block_num, shop_id, item_id`): el mismo objeto sirve para entrenar, validar y predecir `test.csv`, y no usa información futura (verificado).
3. Se guardó ajustado con los meses 0-26 en **`models/preprocessing_pipeline.pkl` (18 MB)**, autocontenido (incluye la historia mensual) y comprobado al cargarlo de nuevo desde disco.

**Siguiente paso:** `04_model_training_n_tunning.ipynb` carga este `.pkl`, entrena y compara 15
modelos (5 familias × 3 configuraciones) con RMSE en validación y agrega el ganador como paso final.